# Kouman Edge 1B (`Tree-AI-lab/kouman-edge-1b`) — Notebook SFT V5 (Style Meta AI Distillé + Traduction Paragraphes + Chat Vector) & Export GGUF (`Q8_0 → Q2_K`) sur AMD ROCm

Entraîne **Kouman Edge 1B** à partir du CPT fusionné (**`issoufzousko07/kouman-llama32-1b-cpt-merged`**, PPL finale `BCI 2.90` / `DYU 2.25` / `FR 4.40` / `EN 5.12`) et du dataset SFT V5 LIMA (**`issoufzousko07/kouman-edge-1b-sft-v5-meta-distille`** — style Meta AI distillé + vrais paragraphes parallèles Cosmopedia/FineWeb + dialogues Baoulé/Dioula vérifiés), puis **remplace directement le modèle actuel** dans le dépôt existant **`Tree-AI-lab/kouman-edge-1b`** (aucun nouveau dépôt créé, `audio_onnx/` préservé) :
1. Le **modèle Instruct 1B complet `bfloat16`** (`safetensors`, natif `llama`, zéro `trust_remote_code`),
2. Toute la gamme **GGUF `llama.cpp` de `Q8_0` jusqu'à `Q2_K`** calibrée **`imatrix` 4 langues** (`bci + dyu + fr + en`) prête pour smartphone hors-ligne !

### Calibrage anti-mémorisation & préservation 100 % CPT (`1,24B` params)
- **Dégel sélectif des 3 seuls tokens Chat Llama 3** (`<|start_header_id|>`, `<|end_header_id|>`, `<|eot_id|>` = `[128006, 128007, 128009]`) : les **128 253 autres embeddings CPT** restent strictement identiques au bit près.
- **Padding officiel Llama 3** : `<|finetune_right_pad_id|>` (`128004`), strictement distinct de `EOS (128001)` et `EOT (128009)`.
- **Chat Vector optionnel (`alpha = 0.65`)** appliqué AVANT LoRA sur le modèle de base brut (couches Transformer + tokens spéciaux Llama 3 `>= 128000`, en gardant 100 % des embeddings CPT lexicaux intacts).
- **LoRA FP32 (`r=32, alpha=64, dropout=0.05`)**, **`lr = 1.5e-4`**, **`3.0` époques max**, masquage `-100` strict sur les tours `user`, Early Stopping toutes les `20` steps.

## 1. Environnement AMD ROCm (`transformers 4.57.3` & Anti-Xet)

In [ ]:
import os

os.environ["PYTORCH_HIP_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["MIOPEN_LOG_LEVEL"] = "3"
os.environ["MIOPEN_ENABLE_LOGGING"] = "0"
os.environ["MIOPEN_ENABLE_LOGGING_CMD"] = "0"
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"

In [ ]:
%pip install --quiet --upgrade pip
%pip install --quiet --force-reinstall --no-cache-dir "transformers==4.57.3" "tokenizers==0.22.2" "huggingface_hub>=0.34.0,<1.0.0"
%pip install --quiet --upgrade-strategy only-if-needed "numpy>=2.0.0,<2.8.0" "pyarrow>=17.0.0" "peft>=0.14.0" "accelerate>=0.34.0" "safetensors" "tqdm"

In [ ]:
import numpy as np

if int(np.__version__.split(".")[0]) < 2:
    raise RuntimeError(
        f"[ERREUR] NumPy {np.__version__} en mémoire : exécutez la cellule %pip install puis redémarrez le kernel (Kernel -> Restart)."
    )

import getpass
import json
import math
import random
import shutil
import time
from pathlib import Path

import peft
import pyarrow.parquet as pq
import torch
import transformers
from huggingface_hub import HfApi, hf_hub_download
from peft import LoraConfig, TaskType, get_peft_model
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

if int(transformers.__version__.split(".")[0]) >= 5:
    raise RuntimeError(
        f"[ERREUR] transformers {transformers.__version__} détecté : redémarrez le kernel après %pip install (<5.0.0)."
    )
if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError("[ERREUR] GPU AMD ROCm / CUDA avec support bfloat16 requis.")

vram_libre_go, vram_totale_go = (x / (1024 ** 3) for x in torch.cuda.mem_get_info(0))
print(f"[OK] GPU actif : {torch.cuda.get_device_name(0)} | VRAM libre : {vram_libre_go:.1f} / {vram_totale_go:.1f} Go")
print(f"transformers={transformers.__version__} | peft={peft.__version__}")

## 2. Configuration SFT Llama 3.2 1B (`LIMA 3 000`) & Authentification HF

In [ ]:
JETON_HF = os.getenv("HF_TOKEN", "").strip()
if not JETON_HF and Path(".env").exists():
    for ligne_env in Path(".env").read_text(encoding="utf-8").splitlines():
        if ligne_env.strip().startswith("HF_TOKEN="):
            JETON_HF = ligne_env.split("=", 1)[1].strip().strip('"').strip("'")
            break
if not JETON_HF:
    JETON_HF = getpass.getpass("Jeton Hugging Face (lecture/écriture issoufzousko07 + Tree-AI-lab) : ").strip()
if not JETON_HF:
    raise PermissionError("[ERREUR] Jeton HF obligatoire.")
os.environ["HF_TOKEN"] = JETON_HF

DEPOT_CPT_FUSIONNE = "issoufzousko07/kouman-llama32-1b-cpt-merged"
DEPOT_DATASET_SFT = "issoufzousko07/kouman-edge-1b-sft-v5-meta-distille"
DEPOT_MODELE_FINAL_SFT = "Tree-AI-lab/kouman-edge-1b"
NOM_COURT_MODELE = "kouman-edge-1b"
CREER_NOUVEAU_DEPOT = False  # Strictement interdit : remplace UNIQUEMENT le modèle dans le dépôt existant

# Hyperparamètres SFT calibrés pour Kouman Edge 1B (Llama 3.2 1B) en LoRA FP32 + 3 tokens Chat FP32 (documents entiers 4096 tok)
LONGUEUR_MAX_SEQ = 4096
TAILLE_MICRO_LOT = 2
PAS_ACCUMULATION = 8                  # Lot effectif conservé = 16 exemples / step (VRAM optimisée pour 4096 tokens)
NB_EPOQUES_MAX = 3.0                  # 3.0 époques max + loss par exemple + val macro sur 9 catégories
TAUX_APPRENTISSAGE = 1.2e-4           # LoRA FP32 avec loss normalisée par exemple (équilibre courts vs documents)
TAUX_APPRENTISSAGE_TOKENS_CHAT = 5.0e-4  # 3 tokens chat [128006, 128007, 128009] en maître FP32
TAUX_WARMUP = 0.08
WEIGHT_DECAY = 0.01
CLIPPING_NORME = 1.0
INTERVALLE_EVAL_STEPS = 20
PATIENCE_EARLY_STOP = 6
NEFTUNE_ALPHA = 0.0                   # Désactivé sur 1B / 3k (évite de bruiter l'identité et EOT)
GRAINE = 42

LORA_R = 16                           # 11,3M params FP32 (~0,90 % du 1.24B — préserve le CPT)
LORA_ALPHA = 32
LORA_DROPOUT = 0.10
MODULES_LORA = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

DOSSIER_SFT = Path("travail_sft_kouman_edge_1b_v5_meta")
ACTIVER_CHAT_VECTOR = False  # Désactivé : le CPT pur (val init 1.90 vs 2.82 avec Chat Vector) est nettement meilleur
ALPHA_CHAT_VECTOR = 0.85     # Optimal Chat Vector (Huang et al.) + LoRA doux pour préserver raisonnement Instruct & CPT
MODELE_LLAMA_BASE = "meta-llama/Llama-3.2-1B"
MODELE_LLAMA_INSTRUCT = "meta-llama/Llama-3.2-1B-Instruct"
DOSSIER_SFT.mkdir(parents=True, exist_ok=True)
DOSSIER_MEILLEUR_ADAPTATEUR = DOSSIER_SFT / "meilleur_adaptateur_sft"
DOSSIER_MODELE_FUSIONNE = DOSSIER_SFT / "kouman_edge_1b_instruct_bf16"
DOSSIER_GGUF = DOSSIER_SFT / "gguf_export"

client_hf = HfApi(token=JETON_HF)
info_depot_cible = client_hf.repo_info(repo_id=DEPOT_MODELE_FINAL_SFT, repo_type="model")
print(f"Base CPT : {DEPOT_CPT_FUSIONNE}")
print(f"Dataset  : {DEPOT_DATASET_SFT}")
print(f"Remplacement direct dans le dépôt existant : https://huggingface.co/{DEPOT_MODELE_FINAL_SFT} (privé={info_depot_cible.private} | aucun nouveau dépôt créé)")

## 3. Template Chat Llama 3 Natif, Chargement Dataset `3k` & Masquage Exact (`labels = -100`)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(DEPOT_CPT_FUSIONNE, token=JETON_HF)

# Pad token officiel Llama 3 (<|finetune_right_pad_id|> = 128004), strictement distinct de BOS/EOS/EOT
ID_PAD_LLAMA3 = tokenizer.convert_tokens_to_ids("<|finetune_right_pad_id|>")
if isinstance(ID_PAD_LLAMA3, int) and ID_PAD_LLAMA3 >= 0 and ID_PAD_LLAMA3 != tokenizer.unk_token_id:
    tokenizer.pad_token = "<|finetune_right_pad_id|>"
    tokenizer.pad_token_id = ID_PAD_LLAMA3
else:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

TEMPLATE_CHAT_LLAMA3_KOUMAN = (
    "{{ bos_token }}"
    "{% for message in messages %}"
    "<|start_header_id|>{{ message['role'] }}<|end_header_id|>\n\n"
    "{{ message['content'] | trim }}<|eot_id|>"
    "{% endfor %}"
    "{% if add_generation_prompt %}<|start_header_id|>assistant<|end_header_id|>\n\n{% endif %}"
)
tokenizer.chat_template = TEMPLATE_CHAT_LLAMA3_KOUMAN

ID_BOS = tokenizer.bos_token_id
ID_EOS = tokenizer.eos_token_id
ID_START_HEADER = tokenizer.convert_tokens_to_ids("<|start_header_id|>")
ID_END_HEADER = tokenizer.convert_tokens_to_ids("<|end_header_id|>")
ID_EOT = tokenizer.convert_tokens_to_ids("<|eot_id|>")

for nom_tok, val_tok in [
    ("BOS", ID_BOS),
    ("EOS", ID_EOS),
    ("<|start_header_id|>", ID_START_HEADER),
    ("<|end_header_id|>", ID_END_HEADER),
    ("<|eot_id|>", ID_EOT),
]:
    if not isinstance(val_tok, int) or val_tok < 0 or val_tok == tokenizer.unk_token_id:
        raise ValueError(f"[ERREUR] Token spécial Llama 3 invalide : {nom_tok}={val_tok}")

assert tokenizer.pad_token_id != ID_EOT, "[ERREUR] pad_token_id ne doit pas être égal à <|eot_id|> !"
IDS_ARRET = [ID_EOS, ID_EOT]
print(
    f"[OK] Tokens spéciaux Llama 3 vérifiés : BOS={ID_BOS}, EOS={ID_EOS}, PAD={tokenizer.pad_token_id}, "
    f"start_header={ID_START_HEADER}, end_header={ID_END_HEADER}, eot={ID_EOT}"
)


def charger_parquets_sft(nom_split: str) -> list[dict]:
    chemin = hf_hub_download(
        repo_id=DEPOT_DATASET_SFT,
        filename=f"data/{nom_split}-00000-of-00001.parquet",
        repo_type="dataset",
        token=JETON_HF,
        force_download=True,
    )
    table = pq.read_table(chemin, columns=["messages", "categorie"])
    return table.to_pylist()


class DatasetChatLlama3SFT(Dataset):
    """Encode les dialogues au format Llama 3 officiel avec masquage (-100) strict sur tous les tokens user/headers."""

    def __init__(self, lignes: list[dict], max_len: int):
        self.exemples: list[tuple[torch.Tensor, torch.Tensor, str]] = []
        ignores = 0

        tok_user = tokenizer.encode("user", add_special_tokens=False)
        tok_asst = tokenizer.encode("assistant", add_special_tokens=False)
        tok_2nl = tokenizer.encode("\n\n", add_special_tokens=False)

        for item in lignes:
            msgs = item["messages"]
            if len(msgs) < 2 or len(msgs) % 2 != 0:
                raise ValueError(f"[ERREUR] Format messages inattendu (nombre impair/vide) : {msgs}")

            ids_complets: list[int] = [ID_BOS]
            labels_complets: list[int] = [-100]
            nb_tours = len(msgs) // 2

            for idx_tour in range(nb_tours):
                m_user = msgs[2 * idx_tour]
                m_asst = msgs[2 * idx_tour + 1]
                if m_user["role"] != "user" or m_asst["role"] != "assistant":
                    raise ValueError(f"[ERREUR] Alternance user/assistant invalide : {msgs}")

                bloc_user_et_header_asst = (
                    [ID_START_HEADER]
                    + tok_user
                    + [ID_END_HEADER]
                    + tok_2nl
                    + tokenizer.encode(m_user["content"].strip(), add_special_tokens=False)
                    + [ID_EOT]
                    + [ID_START_HEADER]
                    + tok_asst
                    + [ID_END_HEADER]
                    + tok_2nl
                )
                bloc_asst = tokenizer.encode(m_asst["content"].strip(), add_special_tokens=False) + [ID_EOT]

                ids_complets.extend(bloc_user_et_header_asst)
                labels_complets.extend([-100] * len(bloc_user_et_header_asst))
                ids_complets.extend(bloc_asst)
                labels_complets.extend(bloc_asst)

            ids_complets = ids_complets[:max_len]
            labels_complets = labels_complets[:max_len]
            if sum(1 for x in labels_complets if x != -100) < 3:
                ignores += 1
                continue

            self.exemples.append(
                (
                    torch.tensor(ids_complets, dtype=torch.long),
                    torch.tensor(labels_complets, dtype=torch.long),
                    item.get("categorie", "inconnue"),
                )
            )

        if not self.exemples:
            raise RuntimeError("[ERREUR] Aucun exemple SFT valide après encodage.")
        print(f"   • {len(self.exemples):,} exemples encodés ({ignores} trop longs ignorés)")

    def __len__(self) -> int:
        return len(self.exemples)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor, str]:
        return self.exemples[idx]


def assembler_lot_padding(batch: list[tuple[torch.Tensor, torch.Tensor, str]]) -> dict:
    seqs_ids, seqs_labels, cats = zip(*batch)
    input_ids = pad_sequence(seqs_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    labels = pad_sequence(seqs_labels, batch_first=True, padding_value=-100)
    attention_mask = (input_ids != tokenizer.pad_token_id).long()
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels, "categories": list(cats)}


print("Téléchargement et tokenisation du dataset SFT Llama 3.2 1B (3k)...")
dataset_train = DatasetChatLlama3SFT(charger_parquets_sft("train"), LONGUEUR_MAX_SEQ)
dataset_val = DatasetChatLlama3SFT(charger_parquets_sft("validation"), LONGUEUR_MAX_SEQ)

# Vérification stricte : l'encodage manuel est identique à apply_chat_template()
_ex_test_ids, _, _ = dataset_train.exemples[0]
_ex_brut = charger_parquets_sft("train")[0]["messages"]
_ex_tpl = tokenizer.encode(
    tokenizer.apply_chat_template(_ex_brut, tokenize=False, add_generation_prompt=False),
    add_special_tokens=False,
)[:LONGUEUR_MAX_SEQ]
assert _ex_test_ids.tolist() == _ex_tpl, "[ERREUR] Écart détecté entre DatasetChatLlama3SFT et apply_chat_template !"
print("[OK] Cohérence 100 % bit-à-bit vérifiée entre DatasetChatLlama3SFT et tokenizer.apply_chat_template().")

g = torch.Generator().manual_seed(GRAINE)
chargeur_train = DataLoader(
    dataset_train, batch_size=TAILLE_MICRO_LOT, shuffle=True, generator=g,
    collate_fn=assembler_lot_padding, pin_memory=True,
)
chargeur_val = DataLoader(
    dataset_val, batch_size=TAILLE_MICRO_LOT, shuffle=False,
    collate_fn=assembler_lot_padding, pin_memory=True,
)

## 3B. Audit Visuel des Extraits du Jeu d'Entraînement (Masquage `-100` avant GPU)

In [ ]:
import random as _rnd
from collections import defaultdict as _dd

par_cat_visu = _dd(list)
for idx_v, (ids_t, lab_t, cat_t) in enumerate(dataset_train.exemples):
    par_cat_visu[cat_t].append((idx_v, ids_t, lab_t))

rng_v = _rnd.Random(GRAINE)
print("=" * 78)
print(f"AUDIT VISUEL SFT LLAMA 3.2 1B ({len(dataset_train):,} train | {len(dataset_val):,} val)")
print("=" * 78)
for cat_v in sorted(par_cat_visu):
    liste_v = par_cat_visu[cat_v]
    print(f"\nCatégorie : {cat_v} ({len(liste_v):,} exemples en train)")
    for idx_v, ids_t, lab_t in rng_v.sample(liste_v, min(2, len(liste_v))):
        nb_masques = int((lab_t == -100).sum().item())
        nb_sup = int((lab_t != -100).sum().item())
        ids_sup = [int(x) for x in lab_t.tolist() if x != -100]
        texte_complet = tokenizer.decode(ids_t.tolist(), skip_special_tokens=False)
        texte_sup = tokenizer.decode(ids_sup, skip_special_tokens=False)
        print(f"  • Exemple #{idx_v} | total={len(ids_t)} tok | masqués(-100)={nb_masques} | supervisés={nb_sup}")
        print(f"    [PROMPT+REP] : {texte_complet[:420]}{'...' if len(texte_complet) > 420 else ''}")
        print(f"    [SUPERVISÉ]  : {texte_sup[:320]}{'...' if len(texte_sup) > 320 else ''}")

## 4. Chargement CPT Llama 3.2 1B (`sdpa`), LoRA Doux & Dégel Sélectif `[128006, 128007, 128009]`

In [ ]:
print(f"Chargement de {DEPOT_CPT_FUSIONNE} en bfloat16 (attn=sdpa)...")
modele_base = AutoModelForCausalLM.from_pretrained(
    DEPOT_CPT_FUSIONNE,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    token=JETON_HF,
).to(device="cuda", dtype=torch.bfloat16)
modele_base.config.use_cache = False

def appliquer_chat_vector_sur_base(modele_cible):
    """Applique CPT + alpha * (Llama-3.2-1B-Instruct - Llama-3.2-1B-Base) sur le modèle brut AVANT LoRA/PEFT.
    Préserve 100 % des embeddings CPT lexicaux (< 128000) et transfère les poids Transformer + tokens Chat (>= 128000).
    """
    if not ACTIVER_CHAT_VECTOR:
        return 0
    import gc
    print(f"Application du Chat Vector (alpha={ALPHA_CHAT_VECTOR}) sur le modèle de base...")
    sd_cpt = modele_cible.state_dict()
    mod_inst = AutoModelForCausalLM.from_pretrained(
        MODELE_LLAMA_INSTRUCT, torch_dtype=torch.float32, device_map="cpu", token=JETON_HF
    )
    mod_base = AutoModelForCausalLM.from_pretrained(
        MODELE_LLAMA_BASE, torch_dtype=torch.float32, device_map="cpu", token=JETON_HF
    )
    sd_inst = mod_inst.state_dict()
    sd_base = mod_base.state_dict()
    nb_fusionnes = 0
    with torch.no_grad():
        for k, param_cpt in sd_cpt.items():
            if k not in sd_inst or k not in sd_base or param_cpt.shape != sd_inst[k].shape:
                continue
            if "embed_tokens" in k or "lm_head" in k:
                # Ne transfère le delta que sur les tokens spéciaux Llama 3 (>= 128000) pour garder le CPT lexical intact
                if param_cpt.shape[0] > 128000:
                    delta_spec = (sd_inst[k][128000:] - sd_base[k][128000:]).to(device=param_cpt.device, dtype=torch.float32)
                    param_cpt[128000:].copy_((param_cpt[128000:].float() + ALPHA_CHAT_VECTOR * delta_spec).to(param_cpt.dtype))
                    nb_fusionnes += 1
            else:
                delta = (sd_inst[k] - sd_base[k]).to(device=param_cpt.device, dtype=torch.float32)
                param_cpt.copy_((param_cpt.float() + ALPHA_CHAT_VECTOR * delta).to(param_cpt.dtype))
                nb_fusionnes += 1
    del mod_inst, mod_base, sd_inst, sd_base
    gc.collect()
    if nb_fusionnes < 100:
        raise RuntimeError(
            f"[ERREUR] Échec Chat Vector : seulement {nb_fusionnes} tenseurs fusionnés (attendu >= 100). "
            "Vérifiez que la fonction est appelée sur le modèle brut AVANT get_peft_model()."
        )
    print(f"[OK] Chat Vector appliqué sur {nb_fusionnes} tenseurs (CPT lexical Baoulé/Dioula 100 % préservé + fluidité Instruct transférée !)")
    return nb_fusionnes

appliquer_chat_vector_sur_base(modele_base)

vram_libre_go = torch.cuda.mem_get_info(0)[0] / (1024 ** 3)
if vram_libre_go < 60.0 or LONGUEUR_MAX_SEQ >= 4096:
    print(f"Activation Gradient Checkpointing (VRAM libre={vram_libre_go:.1f} Go, seq_max={LONGUEUR_MAX_SEQ}).")
    modele_base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    modele_base.enable_input_require_grads()

config_lora_sft = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    use_rslora=False,
    bias="none",
    target_modules=MODULES_LORA,
    task_type=TaskType.CAUSAL_LM,
)
# IMPORTANT : NE PAS faire .to(dtype=torch.bfloat16) après get_peft_model !
# Les poids LoRA (lora_A / lora_B) DOIVENT rester en float32 pour éviter l'underflow d'arrondi bfloat16 sous AdamW.
modele = get_peft_model(modele_base, config_lora_sft)
for nom_p, p in modele.named_parameters():
    if "lora_" in nom_p and p.requires_grad:
        p.data = p.data.to(device="cuda", dtype=torch.float32)

nb_lora_non_fp32 = sum(1 for n, p in modele.named_parameters() if "lora_" in n and p.dtype != torch.float32)
assert nb_lora_non_fp32 == 0, f"[ERREUR] {nb_lora_non_fp32} tenseur(s) LoRA ne sont pas en float32 !"

# Dégel SÉLECTIF en maître FP32 des 3 seuls tokens Chat Llama 3 : <|start_header_id|>, <|end_header_id|>, <|eot_id|>
IDS_SPECIAUX_SFT = [ID_START_HEADER, ID_END_HEADER, ID_EOT]
poids_embed = modele.get_input_embeddings().weight
poids_embed.requires_grad_(True)
maitre_tokens_chat_fp32 = torch.nn.Parameter(
    poids_embed.data[IDS_SPECIAUX_SFT].detach().to(device="cuda", dtype=torch.float32).clone()
)
ID_TEMOIN_CPT = 5000
temoin_emb_avant = poids_embed[ID_TEMOIN_CPT].detach().clone().cpu()
masque_grad_speciaux = torch.zeros(poids_embed.size(0), 1, device=poids_embed.device, dtype=poids_embed.dtype)
masque_grad_speciaux[IDS_SPECIAUX_SFT] = 1.0
handle_masque_emb = poids_embed.register_hook(lambda grad: grad * masque_grad_speciaux)
nb_autres_emb = poids_embed.size(0) - len(IDS_SPECIAUX_SFT)
print(f"LoRA r={LORA_R} en float32 pur + maître FP32 sur 3 tokens chat uniquement : {IDS_SPECIAUX_SFT} ({nb_autres_emb:,} autres embeddings CPT strictement gelés)")


def hook_neftune(module, _entrees, sortie):
    if module.training and NEFTUNE_ALPHA > 0:
        dims = torch.tensor(sortie.size(1) * sortie.size(2), device=sortie.device, dtype=sortie.dtype)
        mag_norm = NEFTUNE_ALPHA / torch.sqrt(dims)
        return sortie + torch.empty_like(sortie).uniform_(-1.0, 1.0) * mag_norm
    return sortie


handle_neftune = modele.get_input_embeddings().register_forward_hook(hook_neftune)

params_lora_count = sum(p.numel() for n, p in modele.named_parameters() if p.requires_grad and p is not poids_embed)
params_emb_spec = len(IDS_SPECIAUX_SFT) * poids_embed.size(1)
tot = sum(p.numel() for p in modele.parameters())
print(f"Paramètres SFT effectifs : {params_lora_count:,} (LoRA r={LORA_R}) + {params_emb_spec:,} (3 tokens chat) / {tot:,} ({100 * (params_lora_count + params_emb_spec) / tot:.2f} % | NEFTune={NEFTUNE_ALPHA})")

## 5. Boucle SFT Llama 3.2 1B avec `val_loss` par Catégorie & Early Stopping

In [ ]:
from collections import defaultdict
import torch.nn.functional as F

NB_EPOQUES_MAX = 3.0
TAUX_APPRENTISSAGE = 1.2e-4

steps_par_epoque = math.ceil(len(chargeur_train) / PAS_ACCUMULATION)
total_steps_max = int(round(steps_par_epoque * NB_EPOQUES_MAX))
steps_warmup = max(10, int(total_steps_max * TAUX_WARMUP))

params_lora = [p for n, p in modele.named_parameters() if p.requires_grad and p is not poids_embed]
optimiseur = torch.optim.AdamW(
    [
        {"params": params_lora, "lr": TAUX_APPRENTISSAGE, "lr_base": TAUX_APPRENTISSAGE, "weight_decay": WEIGHT_DECAY},
        {"params": [maitre_tokens_chat_fp32], "lr": TAUX_APPRENTISSAGE_TOKENS_CHAT, "lr_base": TAUX_APPRENTISSAGE_TOKENS_CHAT, "weight_decay": 0.0},
    ],
    betas=(0.9, 0.99),
)


def sauvegarder_meilleur_sft(dossier_cible: Path) -> None:
    """Sauvegarde l'adaptateur LoRA et les vecteurs FP32 des 3 tokens chat."""
    modele.save_pretrained(str(dossier_cible))
    torch.save(
        {
            "ids": list(IDS_SPECIAUX_SFT),
            "weights": maitre_tokens_chat_fp32.detach().clone().cpu(),
        },
        dossier_cible / "tokens_speciaux_sft.pt",
    )


def facteur_lr_cosine(step_actuel: int) -> float:
    """Calcule le multiplicateur cosine avec warmup linéaire initial."""
    if step_actuel < steps_warmup:
        return float(step_actuel + 1) / float(steps_warmup)
    progression = min(1.0, (step_actuel - steps_warmup) / float(max(1, total_steps_max - steps_warmup)))
    return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progression))


def calculer_perte_par_exemple(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """Calcule la cross-entropy moyenne par exemple afin d'équilibrer exemples courts et documents entiers."""
    shift_logits = logits[:, :-1, :]
    shift_labels = labels[:, 1:]
    pertes_exemples: list[torch.Tensor] = []
    for i in range(shift_labels.size(0)):
        masque_i = shift_labels[i] != -100
        if masque_i.any():
            perte_i = F.cross_entropy(
                shift_logits[i][masque_i].float(),
                shift_labels[i][masque_i],
                reduction="mean",
            )
            pertes_exemples.append(perte_i)
    if not pertes_exemples:
        raise ValueError("[ERREUR] Micro-lot sans aucun token assistant valide (!= -100).")
    return torch.stack(pertes_exemples).mean()


@torch.no_grad()
def evaluer_val_loss() -> tuple[float, dict[str, float]]:
    """Évalue la validation en macro-moyenne sur les 9 catégories (poids égal par compétence)."""
    modele.eval()
    somme_cat: dict[str, float] = defaultdict(float)
    nb_cat: dict[str, int] = defaultdict(int)
    for lot in chargeur_val:
        cats = lot["categories"]
        lot_cuda = {k: v.to("cuda", non_blocking=True) for k, v in lot.items() if k != "categories"}
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            logits = modele(input_ids=lot_cuda["input_ids"], attention_mask=lot_cuda["attention_mask"]).logits
        shift_logits = logits[:, :-1, :]
        shift_labels = lot_cuda["labels"][:, 1:]
        for i, cat in enumerate(cats):
            masque_i = shift_labels[i] != -100
            if masque_i.any():
                perte_ex = float(
                    F.cross_entropy(
                        shift_logits[i][masque_i].float(),
                        shift_labels[i][masque_i],
                        reduction="mean",
                    ).item()
                )
                somme_cat[cat] += perte_ex
                nb_cat[cat] += 1
    torch.cuda.empty_cache()
    modele.train()
    if not nb_cat:
        raise RuntimeError("[ERREUR] Validation vide : aucune catégorie évaluée.")
    par_cat = {c: round(somme_cat[c] / nb_cat[c], 4) for c in sorted(somme_cat)}
    val_macro = sum(somme_cat[c] / nb_cat[c] for c in somme_cat) / len(somme_cat)
    return val_macro, par_cat


val_loss_init, val_cats_init = evaluer_val_loss()
print(f"Validation initiale avant SFT : val_macro={val_loss_init:.4f} | par catégorie={val_cats_init}")

meilleure_val_loss = val_loss_init
sans_amelioration = 0
step_global = 0
arret_anticipe = False
au_moins_une_sauvegarde = False
historique_sft = []
fenetre_train: list[float] = []

modele.train()
optimiseur.zero_grad(set_to_none=True)
compteur_accum = 0
perte_accum = 0.0

barre = tqdm(total=total_steps_max, desc="SFT V5.7 Loss/Exemple + Val Macro (Llama 3.2 1B)")
iterateur = iter(chargeur_train)

while step_global < total_steps_max and not arret_anticipe:
    try:
        lot = next(iterateur)
    except StopIteration:
        iterateur = iter(chargeur_train)
        lot = next(iterateur)

    lot_cuda = {k: v.to("cuda", non_blocking=True) for k, v in lot.items() if k != "categories"}
    with torch.amp.autocast("cuda", dtype=torch.bfloat16):
        logits = modele(input_ids=lot_cuda["input_ids"], attention_mask=lot_cuda["attention_mask"]).logits
        perte = calculer_perte_par_exemple(logits, lot_cuda["labels"]) / PAS_ACCUMULATION

    perte.backward()
    perte_accum += float(perte.detach().cpu()) * PAS_ACCUMULATION
    compteur_accum += 1

    if compteur_accum >= PAS_ACCUMULATION:
        mult = facteur_lr_cosine(step_global)
        for grp in optimiseur.param_groups:
            grp["lr"] = grp["lr_base"] * mult

        # Transfert exact du gradient des 3 tokens chat vers la copie maître float32
        if poids_embed.grad is not None:
            maitre_tokens_chat_fp32.grad = poids_embed.grad[IDS_SPECIAUX_SFT].detach().to(dtype=torch.float32).clone()
            poids_embed.grad = None

        torch.nn.utils.clip_grad_norm_(params_lora + [maitre_tokens_chat_fp32], CLIPPING_NORME)
        optimiseur.step()
        with torch.no_grad():
            poids_embed.data[IDS_SPECIAUX_SFT] = maitre_tokens_chat_fp32.data.to(dtype=poids_embed.dtype)
            emb_out_maj = modele.get_output_embeddings().weight
            if emb_out_maj.data_ptr() != poids_embed.data_ptr():
                emb_out_maj.data[IDS_SPECIAUX_SFT] = maitre_tokens_chat_fp32.data.to(dtype=emb_out_maj.dtype)
        optimiseur.zero_grad(set_to_none=True)

        train_loss_step = perte_accum / compteur_accum
        fenetre_train.append(train_loss_step)
        if len(fenetre_train) > INTERVALLE_EVAL_STEPS:
            fenetre_train.pop(0)
        train_moy = sum(fenetre_train) / len(fenetre_train)
        perte_accum = 0.0
        compteur_accum = 0
        step_global += 1
        barre.update(1)
        barre.set_postfix(train_moy=f"{train_moy:.3f}", best_val=f"{meilleure_val_loss:.3f}")

        if step_global % INTERVALLE_EVAL_STEPS == 0 or step_global == total_steps_max:
            val_macro, val_cats = evaluer_val_loss()
            historique_sft.append({
                "step": step_global,
                "train_moy": round(train_moy, 4),
                "val_loss": round(val_macro, 4),
                "val_macro": round(val_macro, 4),
                "val_categories": val_cats,
            })
            if val_macro < meilleure_val_loss - 0.001:
                meilleure_val_loss = val_macro
                sans_amelioration = 0
                sauvegarder_meilleur_sft(DOSSIER_MEILLEUR_ADAPTATEUR)
                au_moins_une_sauvegarde = True
                print(f"\nStep {step_global} | train_moy={train_moy:.4f} | val_macro={val_macro:.4f} (nouveau meilleur [OK]) | {val_cats}")
            else:
                sans_amelioration += 1
                print(f"\n[PAUSE] Step {step_global} | train_moy={train_moy:.4f} | val_macro={val_macro:.4f} (stagnation {sans_amelioration}/{PATIENCE_EARLY_STOP}) | {val_cats}")
                if sans_amelioration >= PATIENCE_EARLY_STOP:
                    print("[ARRET] Arrêt anticipé activé pour préserver la généralisation.")
                    arret_anticipe = True

barre.close()
handle_neftune.remove()
handle_masque_emb.remove()
assert torch.equal(modele.get_input_embeddings().weight[ID_TEMOIN_CPT].detach().cpu(), temoin_emb_avant), "[ERREUR] Erreur : un embedding CPT non-spécial a dérivé !"
print(f"[OK] Contrôle bit-à-bit OK : les {nb_autres_emb:,} embeddings CPT sont restés strictement identiques.")
if not au_moins_une_sauvegarde or not DOSSIER_MEILLEUR_ADAPTATEUR.exists():
    sauvegarder_meilleur_sft(DOSSIER_MEILLEUR_ADAPTATEUR)
print(f"SFT V5 Llama 3.2 1B terminé | meilleur val_macro={meilleure_val_loss:.4f} | adaptateur={DOSSIER_MEILLEUR_ADAPTATEUR}")


## 6. Tests OOD Ciblés (Anti-Auto-Présentation Dioula/Baoulé/FR, ASR Minuscules, Multi-Tour 4–5 Tours)

In [ ]:
from peft import PeftModel

del modele, modele_base
torch.cuda.empty_cache()
modele_base_propre = AutoModelForCausalLM.from_pretrained(
    DEPOT_CPT_FUSIONNE,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    token=JETON_HF,
)
appliquer_chat_vector_sur_base(modele_base_propre)
chemin_spec = DOSSIER_MEILLEUR_ADAPTATEUR / "tokens_speciaux_sft.pt"
if not chemin_spec.exists():
    raise FileNotFoundError(f"[ERREUR] Vecteurs tokens spéciaux introuvables : {chemin_spec}")
etat_spec = torch.load(chemin_spec, map_location="cpu", weights_only=True)
with torch.no_grad():
    emb_in = modele_base_propre.get_input_embeddings().weight
    emb_in[etat_spec["ids"]] = etat_spec["weights"].to(dtype=emb_in.dtype)
    emb_out = modele_base_propre.get_output_embeddings().weight
    if emb_out.data_ptr() != emb_in.data_ptr():
        emb_out[etat_spec["ids"]] = etat_spec["weights"].to(dtype=emb_out.dtype)
print(f"[OK] Vecteurs entraînés Chat Llama 3 réinjectés : ids={etat_spec['ids']}")

modele_eval = PeftModel.from_pretrained(
    modele_base_propre,
    str(DOSSIER_MEILLEUR_ADAPTATEUR),
    is_trainable=False,
).to(device="cuda")
modele_eval.eval()

QUESTIONS_OOD_LLAMA = [
    ("FR oral ASR (mécanique — 0 bio !)", "je veux que tu me parles de la mécanique"),
    ("DYU libre ASR (santé — 0 bio !)", "n b'a fɛ i ka kuma n ye kɛnɛya kan"),
    ("DYU libre OOD (encyclopédie)", "Hakili min be sɔrɔ mɔgɔ bolo, o ye mun lo ye?"),
    ("BCI libre ASR (village — 0 bio !)", "kan klɔ su mɛn dilɛ’n i ndɛ kle min"),
    ("Identité croisée OOD (Nicanor)", "bonjour je suis nicanor et toi ?"),
    ("Conception Kouman (OOD)", "raconte moi comment tu as ete conçu"),
    ("Bascule langue OOD (FR -> DYU)", "bonjour repond moi en dioula"),
    ("Double langue OOD (FR + BCI)", "presente toi en français puis en baoulé"),
    ("Traduction FR -> BCI (OOD)", "traduis en baoulé : demain matin je vais saluer mes grands-parents au village"),
    ("Traduction FR -> DYU (OOD)", "traduis en dioula : ma sœur prépare du riz chaud pour le dîner de la famille"),
    ("Calcul 4 chiffres (OOD)", "combien font 2345+5688"),
    ("Traduction Document Entier FR -> BCI (OOD)", "Traduis ce document entier en baoulé sans rien omettre :\n\nAgriculture durable et solidarité rurale en Côte d'Ivoire\n\nL'agriculture demeure le pilier essentiel de la vie quotidienne pour des millions de familles en Côte d'Ivoire. Chaque matin, dès les premières heures, les producteurs se rendent dans leurs plantations pour entretenir les cultures vivrières, le cacao, le café et l'anacarde. Ce travail patient demande une grande attention aux sols, aux pluies et aux saisons.\n\nAu fil des dernières années, les coopératives locales ont renforcé l'entraide entre cultivateurs. Grâce à la mise en commun des outils, au partage de semences sélectionnées et à de meilleures techniques de séchage, la qualité des récoltes progresse régulièrement. Les femmes jouent un rôle majeur dans la transformation du manioc en attiéké et dans l'approvisionnement des marchés urbains et ruraux.\n\nAujourd'hui, l'adaptation aux changements du climat devient une priorité partagée par tous les villages. La protection des jeunes arbres, l'usage raisonné de l'eau et l'association de plusieurs cultures sur une même parcelle permettent de préserver la fertilité des terres pour les générations futures.\n\nEn conclusion, soutenir le travail des agriculteurs ivoiriens garantit à la fois la sécurité alimentaire du pays et la vitalité économique de chaque communauté."),
    ("Traduction Document Entier FR -> DYU (OOD)", "Traduis ce document entier en dioula sans rien omettre :\n\nL'éducation et les outils numériques au service de la jeunesse\n\nL'apprentissage scolaire ouvre de nombreuses portes aux jeunes générations dans toute l'Afrique de l'Ouest. Quand les élèves travaillent avec sérieux à l'école, lisent régulièrement et posent des questions curieuses en classe, ils préparent des bases solides pour comprendre le monde moderne et choisir librement leur métier.\n\nAujourd'hui, les téléphones et les ordinateurs deviennent des compagnons d'étude précieux, même dans les localités éloignées. Avec des applications éducatives accessibles hors connexion, un étudiant peut réviser les sciences, rédiger des exposés clairs et traduire des textes entre le français, le dioula, le baoulé et l'anglais sans dépendre d'une connexion coûteuse.\n\nCependant, la technologie ne remplace jamais l'effort humain ni les conseils des enseignants et des parents. L'entraide entre camarades, le respect des anciens et la discipline personnelle restent indispensables pour transformer ces nouveaux outils en véritables sources de progrès partagé.\n\nEn résumé, unir la rigueur scolaire et le numérique dans nos langues permet aux jeunes de bâtir des projets utiles pour leurs familles et pour l'avenir du pays."),
]

for titre, question in QUESTIONS_OOD_LLAMA:
    prompt_chat = tokenizer.apply_chat_template(
        [{"role": "user", "content": question}],
        tokenize=False,
        add_generation_prompt=True,
    )
    entrees = tokenizer(prompt_chat, return_tensors="pt", add_special_tokens=False).to("cuda")
    torch.manual_seed(GRAINE)
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
        sortie = modele_eval.generate(
            **entrees,
            max_new_tokens=1536,
            do_sample=True,
            temperature=0.4,
            top_p=0.9,
            repetition_penalty=1.08,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=IDS_ARRET,
        )
    gen_ids = sortie[0, entrees["input_ids"].shape[1]:].tolist()
    arret = bool(gen_ids) and gen_ids[-1] in IDS_ARRET
    rep = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    print("=" * 72)
    print(f"{titre} | {'[OK] EOT' if arret else '[ERREUR] MAX'} ({len(gen_ids)} tok)\nUtilisateur : {question}\nAssistant : {rep}")


def executer_test_multitour(nom_scenario: str, tours_user: list[str], max_tours_memoire: int = 4):
    print("=" * 72)
    print(f"MULTI-TOUR OOD — {nom_scenario}")
    hist: list[dict[str, str]] = []
    for idx_u, msg_u in enumerate(tours_user, start=1):
        hist.append({"role": "user", "content": msg_u})
        fenetre = hist[-(max_tours_memoire * 2 + 1):]
        if fenetre[0]["role"] != "user":
            fenetre = fenetre[1:]
        p_mt = tokenizer.apply_chat_template(fenetre, tokenize=False, add_generation_prompt=True)
        entrees_mt = tokenizer(p_mt, return_tensors="pt", add_special_tokens=False).to("cuda")
        torch.manual_seed(GRAINE + idx_u)
        with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
            sortie_mt = modele_eval.generate(
                **entrees_mt,
                max_new_tokens=512,
                do_sample=True,
                temperature=0.45,
                top_p=0.9,
                repetition_penalty=1.08,
                use_cache=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=IDS_ARRET,
            )
        rep_mt = tokenizer.decode(sortie_mt[0, entrees_mt["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        hist.append({"role": "assistant", "content": rep_mt})
        print(f"  [Tour {idx_u}] Utilisateur : {msg_u}\n  [Tour {idx_u}] Assistant : {rep_mt}")


executer_test_multitour(
    "Dioula Conversation + Cours + Changement Sujet (4 tours)",
    [
        "I ni sɔgɔma",
        "Nne ka kɛnɛ, n b'a fɛ ka jatebɔ dege",
        "Jatikalan kalan n ye.",
        "Hakili min be sɔrɔ mɔgɔ bolo, o ye mun lo ye?",
    ],
)

executer_test_multitour(
    "Relances Courtes + Bascule + Identité (5 tours)",
    [
        "Comment on dit bonjour et bonsoir en baoulé et en dioula ?",
        "Comment on dit merci beaucoup en anglais et en dioula ?",
        "et comment on dit je rentre à la maison maintenant en dioula ?",
        "traduis en baoulé : la vie est belle au village",
        "bonjour je suis nicanor et toi ?",
    ],
)

## 7. Fusion Finale Standard `llama` & Remplacement dans `Tree-AI-lab/kouman-edge-1b`

In [ ]:
# Sécurité : passez à True UNIQUEMENT après avoir validé les tests OOD de la cellule 6 !
PUBLIER_MODELE_FINAL = False

if not PUBLIER_MODELE_FINAL:
    print("[PAUSE] Publication désactivée par sécurité (PUBLIER_MODELE_FINAL = False).")
    print("Vérifiez les tests mono-tour et multi-tour ci-dessus, puis passez PUBLIER_MODELE_FINAL = True.")
else:
    os.environ["HF_HUB_DISABLE_XET"] = "1"
    try:
        import huggingface_hub.constants as _hf_const
        import huggingface_hub._commit_api as _hf_commit
        _hf_const.HF_HUB_DISABLE_XET = True
        if hasattr(_hf_commit, "is_xet_available"):
            _hf_commit.is_xet_available = lambda: False
    except Exception:
        pass

    if "modele_eval" not in globals() or not hasattr(modele_eval, "peft_config"):
        from peft import PeftModel
        print("Rechargement propre du meilleur adaptateur + 3 tokens chat avant fusion...")
        modele_base_propre = AutoModelForCausalLM.from_pretrained(
            DEPOT_CPT_FUSIONNE, torch_dtype=torch.bfloat16, attn_implementation="sdpa", token=JETON_HF
        )
        appliquer_chat_vector_sur_base(modele_base_propre)
        etat_spec = torch.load(DOSSIER_MEILLEUR_ADAPTATEUR / "tokens_speciaux_sft.pt", map_location="cpu", weights_only=True)
        with torch.no_grad():
            emb_in = modele_base_propre.get_input_embeddings().weight
            emb_in[etat_spec["ids"]] = etat_spec["weights"].to(dtype=emb_in.dtype)
            emb_out = modele_base_propre.get_output_embeddings().weight
            if emb_out.data_ptr() != emb_in.data_ptr():
                emb_out[etat_spec["ids"]] = etat_spec["weights"].to(dtype=emb_out.dtype)
        modele_eval = PeftModel.from_pretrained(
            modele_base_propre, str(DOSSIER_MEILLEUR_ADAPTATEUR), is_trainable=False
        ).to(device="cuda")

    print("Fusion finale LoRA SFT sur le CPT Llama 3.2 1B...")
    modele_final = modele_eval.merge_and_unload().to(dtype=torch.bfloat16)
    del modele_eval
    modele_final.config.use_cache = True
    modele_final.generation_config.eos_token_id = IDS_ARRET
    modele_final.generation_config.pad_token_id = tokenizer.pad_token_id
    modele_final.generation_config.do_sample = True
    modele_final.generation_config.temperature = 0.45
    modele_final.generation_config.top_p = 0.9
    modele_final.generation_config.repetition_penalty = 1.08

    DOSSIER_MODELE_FUSIONNE.mkdir(parents=True, exist_ok=True)
    # Contourne accelerate -> deepspeed -> distutils manquant sous Python 3.12
    import accelerate.utils as _acc_u
    import accelerate.utils.imports as _acc_imp
    import accelerate.utils.other as _acc_oth
    for _mod_acc in (_acc_u, _acc_imp, _acc_oth):
        if hasattr(_mod_acc, "is_deepspeed_available"):
            _mod_acc.is_deepspeed_available = lambda: False
    modele_final.save_pretrained(str(DOSSIER_MODELE_FUSIONNE), safe_serialization=True, max_shard_size="5GB")
    tokenizer.chat_template = TEMPLATE_CHAT_LLAMA3_KOUMAN
    tokenizer.save_pretrained(str(DOSSIER_MODELE_FUSIONNE))

    (DOSSIER_MODELE_FUSIONNE / "historique_sft_llama32_1b_v5.json").write_text(
        json.dumps(historique_sft, indent=2, ensure_ascii=False), encoding="utf-8"
    )

    code_inference = "\n".join([
        "#!/usr/bin/env python3",
        "# Script d'inférence autonome — Kouman AI Llama 3.2 1B Instruct (Tree AI Lab)",
        "import argparse",
        "import torch",
        "from transformers import AutoModelForCausalLM, AutoTokenizer",
        "",
        f"DEPOT_DEFAUT = {DEPOT_MODELE_FINAL_SFT!r}",
        "",
        "",
        "def charger_kouman(depot: str = DEPOT_DEFAUT, jeton_hf: str | None = None):",
        "    tokenizer = AutoTokenizer.from_pretrained(depot, token=jeton_hf)",
        "    dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else (torch.float16 if torch.cuda.is_available() else torch.float32)",
        "    modele = AutoModelForCausalLM.from_pretrained(",
        "        depot,",
        "        torch_dtype=dtype,",
        "        device_map='auto' if torch.cuda.is_available() else None,",
        "        attn_implementation='sdpa',",
        "        token=jeton_hf,",
        "    )",
        "    modele.eval()",
        "    return modele, tokenizer",
        "",
        "",
        "def repondre_messages(modele, tokenizer, messages: list[dict[str, str]], max_new_tokens: int | None = None, temperature: float = 0.45) -> str:",
        "    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)",
        "    entrees = tokenizer(prompt, return_tensors='pt', add_special_tokens=False).to(modele.device)",
        "    id_eot = tokenizer.convert_tokens_to_ids('<|eot_id|>')",
        "    ids_arret = [i for i in (tokenizer.eos_token_id, id_eot) if isinstance(i, int) and i >= 0]",
        "    limite_ctx = getattr(modele.config, 'max_position_embeddings', 8192)",
        "    tokens_restants = max(1, limite_ctx - entrees['input_ids'].shape[1])",
        "    budget = tokens_restants if max_new_tokens is None else min(max_new_tokens, tokens_restants)",
        "    with torch.no_grad():",
        "        sortie = modele.generate(",
        "            **entrees,",
        "            max_new_tokens=budget,",
        "            do_sample=temperature > 0,",
        "            temperature=max(temperature, 1e-4),",
        "            top_p=0.9,",
        "            repetition_penalty=1.08,",
        "            pad_token_id=tokenizer.pad_token_id,",
        "            eos_token_id=ids_arret,",
        "        )",
        "    return tokenizer.decode(sortie[0, entrees['input_ids'].shape[1]:], skip_special_tokens=True).strip()",
        "",
        "",
        "if __name__ == '__main__':",
        "    p = argparse.ArgumentParser()",
        "    p.add_argument('--modele', default=DEPOT_DEFAUT)",
        "    p.add_argument('--question', required=True)",
        "    args = p.parse_args()",
        "    mod, tok = charger_kouman(args.modele)",
        "    print(repondre_messages(mod, tok, [{'role': 'user', 'content': args.question}]))",
    ]) + "\n"
    (DOSSIER_MODELE_FUSIONNE / "inference_kouman_edge_1b.py").write_text(code_inference, encoding="utf-8")

    # Vérification stricte : AUCUNE création de dépôt, remplacement uniquement dans le dépôt existant
    if not client_hf.repo_exists(repo_id=DEPOT_MODELE_FINAL_SFT, repo_type="model"):
        raise RuntimeError(f"[ERREUR] Le dépôt cible {DEPOT_MODELE_FINAL_SFT} est introuvable (création interdite).")
    client_hf.upload_folder(
        folder_path=str(DOSSIER_MODELE_FUSIONNE),
        repo_id=DEPOT_MODELE_FINAL_SFT,
        repo_type="model",
        commit_message=f"feat(v5): remplacement Kouman Edge 1B Instruct BF16 (SFT V5 Meta+Paragraphes+ChatVector, val_loss={meilleure_val_loss:.4f})",
    )
    print(f"Modèle BF16 remplacé dans le dépôt existant : https://huggingface.co/{DEPOT_MODELE_FINAL_SFT} (audio_onnx/ conservé intact)")

## 8. Vérification Post-Fusion Autonome (sans PEFT, sans `trust_remote_code`)

In [ ]:
import gc

for var_nom in ("modele_eval", "modele_final", "modele_base_propre"):
    if var_nom in globals():
        del globals()[var_nom]
gc.collect()
torch.cuda.empty_cache()

if not DOSSIER_MODELE_FUSIONNE.exists():
    raise FileNotFoundError(
        f"[ERREUR] Le modèle fusionné local '{DOSSIER_MODELE_FUSIONNE}' n'existe pas encore. "
        "Passez PUBLIER_MODELE_FINAL = True en Cellule 7 avant d'exécuter la Cellule 8."
    )

source_fusion = str(DOSSIER_MODELE_FUSIONNE)
print(f"Rechargement autonome standard (sans PEFT, sans trust_remote_code) depuis : {source_fusion}")

tok_fusion = AutoTokenizer.from_pretrained(source_fusion, token=JETON_HF)
mod_fusion = AutoModelForCausalLM.from_pretrained(
    source_fusion,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    token=JETON_HF,
).to("cuda")
mod_fusion.eval()

assert mod_fusion.config.model_type == "llama", f"[ERREUR] model_type inattendu : {mod_fusion.config.model_type}"
nb_nan = sum(int(torch.isnan(p).any().item() or torch.isinf(p).any().item()) for p in mod_fusion.parameters())
assert nb_nan == 0, f"[ERREUR] {nb_nan} tenseur(s) contiennent NaN/Inf après fusion !"
assert tok_fusion.chat_template is not None, "[ERREUR] chat_template manquant !"
id_eot_f = tok_fusion.convert_tokens_to_ids("<|eot_id|>")
ids_arret_f = [i for i in (tok_fusion.eos_token_id, id_eot_f) if isinstance(i, int) and i >= 0]
print(f"[OK] Intégrité OK : model_type='llama' | 0 NaN/Inf | chat_template OK | eos_token_id={mod_fusion.generation_config.eos_token_id}")

TESTS_POST_MERGE = [
    ("FR oral ASR (mécanique)", "je veux que tu me parles de la mécanique"),
    ("DYU libre ASR (santé)", "n b'a fɛ i ka kuma n ye kɛnɛya kan"),
    ("Identité Nicanor (OOD)", "bonjour je suis nicanor et toi ?"),
    ("Bascule FR->DYU (OOD)", "bonjour repond moi en dioula"),
    ("Traduction FR->DYU (OOD)", "traduis en dioula : ma sœur prépare du riz chaud pour le dîner de la famille"),
    ("Traduction FR->BCI (OOD)", "traduis en baoulé : demain matin je vais saluer mes grands-parents au village"),
]

ok_arret = 0
for titre_m, q_m in TESTS_POST_MERGE:
    p_m = tok_fusion.apply_chat_template([{"role": "user", "content": q_m}], tokenize=False, add_generation_prompt=True)
    inp_m = tok_fusion(p_m, return_tensors="pt", add_special_tokens=False).to("cuda")
    torch.manual_seed(GRAINE)
    with torch.no_grad():
        out_m = mod_fusion.generate(**inp_m, max_new_tokens=512)
    gen_ids = out_m[0, inp_m["input_ids"].shape[1]:].tolist()
    arret = bool(gen_ids) and gen_ids[-1] in ids_arret_f
    ok_arret += int(arret)
    rep_m = tok_fusion.decode(gen_ids, skip_special_tokens=True).strip()
    print("=" * 72)
    print(f"[POST-MERGE LLAMA 3.2 1B] {titre_m} | {'[OK] EOT' if arret else '[ERREUR] MAX'} ({len(gen_ids)} tok)")
    print(f"Utilisateur : {q_m}\nAssistant : {rep_m}")

print("=" * 72)
print(f"Validation post-merge Kouman Edge 1B : {ok_arret}/{len(TESTS_POST_MERGE)} arrêts naturels.")

## 9. Quantification GGUF Complète `llama.cpp` (`Q8_0 → Q2_K` + `imatrix` 4 Langues) & Publication dans `Tree-AI-lab/kouman-edge-1b`

Génère automatiquement toutes les versions mobiles/on-device **GGUF** avec **`llama.cpp`** à partir du modèle Instruct BF16 fusionné, calibre une **Importance Matrix (`imatrix`)** sur nos 4 langues (**Baoulé, Dioula, Français, Anglais**) pour protéger la qualité en `Q4` / `Q3` / `Q2`, teste `Q4_K_M` et `Q2_K`, crée la **Model Card (`README.md`)**, et publie l'ensemble directement dans **`Tree-AI-lab/kouman-edge-1b`**.

In [ ]:
import gc
import os
import shutil
import subprocess
import sys
from pathlib import Path

PUBLIER_GGUF_SUR_HF = True
GARDER_GGUF_F16 = False  # Mettre True pour inclure aussi kouman-edge-1b-F16.gguf (~2,48 Go) sur HF
UTILISER_IMATRIX = True  # Recommandé : protège Baoulé & Dioula sur Q4/Q3/Q2
CHUNKS_IMATRIX = 40      # ~1-2 min sur CPU multi-cœurs pour un modèle 1B

# Gamme complète demandée : de Q8_0 jusqu'à Q2_K
QUANTIFICATIONS_GGUF = [
    ("Q8_0", "Quasi sans perte (desktop / flagship 8+ Go RAM)", False),
    ("Q6_K", "Qualité quasi-BF16, excellent compromis", True),
    ("Q5_K_M", "Haute qualité (smartphones haut de gamme / PC)", True),
    ("Q5_K_S", "Haute qualité compacte", True),
    ("Q4_K_M", "Recommandé mobile Android / iOS (équilibre qualité/vitesse)", True),
    ("Q4_K_S", "Mobile compact 4 bits", True),
    ("Q4_0", "Mobile ultra-rapide CPU ARM (NEON / i8mm)", True),
    ("Q3_K_M", "Smartphones entrée de gamme (4 Go RAM)", True),
    ("Q3_K_S", "Très compact 3 bits", True),
    ("Q2_K", "Ultra-léger 2 bits (anciens smartphones / RAM très limitée)", True),
]

# Libère la VRAM avant compilation / quantification CPU
for var_gpu in ("mod_fusion", "modele", "modele_eval", "modele_final"):
    if var_gpu in globals():
        del globals()[var_gpu]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Vérifie que le modèle Instruct BF16 fusionné est disponible localement (sinon télécharge depuis HF)
if not DOSSIER_MODELE_FUSIONNE.exists() or not (DOSSIER_MODELE_FUSIONNE / "config.json").exists():
    if "PUBLIER_MODELE_FINAL" in globals() and not PUBLIER_MODELE_FINAL:
        raise RuntimeError(
            "[PAUSE] Le modèle Instruct BF16 n'est pas encore fusionné : validez les tests OOD (Cellule 6), "
            "passez PUBLIER_MODELE_FINAL = True en Cellule 7, puis relancez Cellules 7, 8 et 9."
        )
    print(f"Téléchargement du modèle fusionné depuis {DEPOT_MODELE_FINAL_SFT}...")
    from huggingface_hub import snapshot_download
    DOSSIER_MODELE_FUSIONNE.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id=DEPOT_MODELE_FINAL_SFT,
        repo_type="model",
        local_dir=str(DOSSIER_MODELE_FUSIONNE),
        ignore_patterns=["*.gguf", "*.imatrix"],
        token=JETON_HF,
    )


def executer_cmd(cmd: list[str], cwd: Path | None = None) -> None:
    print("", " ".join(str(x) for x in cmd))
    res = subprocess.run(cmd, cwd=str(cwd) if cwd else None, text=True)
    if res.returncode != 0:
        raise RuntimeError(f"[ERREUR] Échec commande (code {res.returncode}) : {' '.join(str(x) for x in cmd)}")


# 1. Prépare cmake + gguf + compilation ciblée llama.cpp (CPU natif ultra-fiable, < 1 min)
if shutil.which("cmake") is None:
    executer_cmd([sys.executable, "-m", "pip", "install", "--quiet", "cmake", "ninja"])

DOSSIER_LLAMA_CPP = DOSSIER_SFT / "llama.cpp"
if not DOSSIER_LLAMA_CPP.exists():
    executer_cmd(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp.git", str(DOSSIER_LLAMA_CPP)])

executer_cmd([sys.executable, "-m", "pip", "install", "--quiet", "gguf", "sentencepiece", "protobuf"])

DOSSIER_BUILD_LLAMA = DOSSIER_LLAMA_CPP / "build"
nb_coeurs = str(max(2, os.cpu_count() or 4))
bin_quantize = DOSSIER_BUILD_LLAMA / "bin" / "llama-quantize"
bin_imatrix = DOSSIER_BUILD_LLAMA / "bin" / "llama-imatrix"
bin_cli = DOSSIER_BUILD_LLAMA / "bin" / "llama-cli"

if not (bin_quantize.exists() and bin_imatrix.exists() and bin_cli.exists()):
    executer_cmd([
        "cmake", "-B", str(DOSSIER_BUILD_LLAMA), "-S", str(DOSSIER_LLAMA_CPP),
        "-DCMAKE_BUILD_TYPE=Release",
        "-DLLAMA_CURL=OFF",
        "-DBUILD_SHARED_LIBS=OFF",
    ])
    executer_cmd([
        "cmake", "--build", str(DOSSIER_BUILD_LLAMA),
        "--config", "Release",
        "--target", "llama-quantize", "llama-imatrix", "llama-cli",
        "-j", nb_coeurs,
    ])

for binaire in (bin_quantize, bin_imatrix, bin_cli):
    if not binaire.exists():
        raise FileNotFoundError(f"[ERREUR] Binaire llama.cpp introuvable après build : {binaire}")
print(f"[OK] Binaires llama.cpp prêts ({nb_coeurs} cœurs CPU).")

# 2. Conversion Hugging Face BF16 -> GGUF F16 maître (réutilise le fichier s'il existe déjà)
DOSSIER_GGUF.mkdir(parents=True, exist_ok=True)
chemin_gguf_f16 = DOSSIER_GGUF / f"{NOM_COURT_MODELE}-F16.gguf"
script_conv = DOSSIER_LLAMA_CPP / "convert_hf_to_gguf.py"
if not (chemin_gguf_f16.exists() and chemin_gguf_f16.stat().st_size > 500_000_000):
    executer_cmd([
        sys.executable, str(script_conv),
        str(DOSSIER_MODELE_FUSIONNE),
        "--outfile", str(chemin_gguf_f16),
        "--outtype", "f16",
    ])
assert chemin_gguf_f16.exists() and chemin_gguf_f16.stat().st_size > 500_000_000, "[ERREUR] Conversion F16 GGUF invalide !"
print(f"[OK] GGUF F16 maître prêt : {chemin_gguf_f16.stat().st_size / (1024**3):.2f} Go")

# 3. Calibration imatrix équilibrée sur nos 4 langues (Baoulé, Dioula, Français, Anglais)
chemin_imatrix = DOSSIER_GGUF / f"{NOM_COURT_MODELE}.imatrix"
if UTILISER_IMATRIX:
    chemin_calib = DOSSIER_SFT / "calibration_imatrix_4langues.txt"
    blocs_calib = []
    for nom_split in ("validation", "train"):
        p_parq = Path(hf_hub_download(
            repo_id=DEPOT_DATASET_SFT,
            filename=f"data/{nom_split}-00000-of-00001.parquet",
            repo_type="dataset",
            token=JETON_HF,
        ))
        tbl = pq.read_table(p_parq, columns=["messages"])
        for conv in tbl["messages"].to_pylist():
            texte_conv = "\n".join(m["content"] for m in conv if isinstance(m, dict) and m.get("content"))
            if texte_conv:
                blocs_calib.append(texte_conv)
            if len(blocs_calib) >= 320:
                break
        if len(blocs_calib) >= 320:
            break
    if not blocs_calib:
        raise RuntimeError("[ERREUR] Corpus de calibration imatrix vide !")
    chemin_calib.write_text("\n\n".join(blocs_calib), encoding="utf-8")
    print(f"Calcul imatrix 4 langues ({len(blocs_calib)} conversations, {CHUNKS_IMATRIX} chunks)...")
    executer_cmd([
        str(bin_imatrix),
        "-m", str(chemin_gguf_f16),
        "-f", str(chemin_calib),
        "-o", str(chemin_imatrix),
        "--chunks", str(CHUNKS_IMATRIX),
        "-t", nb_coeurs,
        "-ngl", "0",
    ])
    assert chemin_imatrix.exists() and chemin_imatrix.stat().st_size > 100_000, "[ERREUR] Fichier imatrix invalide !"
    print(f"[OK] imatrix 4 langues générée : {chemin_imatrix.stat().st_size / (1024**2):.1f} Mo")

# 4. Quantification complète Q8_0 -> Q2_K
inventaire_gguf = []
for q_type, q_desc, q_avec_imatrix in QUANTIFICATIONS_GGUF:
    nom_fic = f"{NOM_COURT_MODELE}-{q_type}.gguf"
    chemin_out = DOSSIER_GGUF / nom_fic
    cmd_q = [str(bin_quantize)]
    if UTILISER_IMATRIX and q_avec_imatrix and chemin_imatrix.exists():
        cmd_q.extend(["--imatrix", str(chemin_imatrix)])
    cmd_q.extend([str(chemin_gguf_f16), str(chemin_out), q_type, nb_coeurs])
    executer_cmd(cmd_q)
    taille_octets = chemin_out.stat().st_size
    assert taille_octets > 200_000_000, f"[ERREUR] Fichier GGUF anormalement petit : {nom_fic} ({taille_octets} octets)"
    inventaire_gguf.append({
        "fichier": nom_fic,
        "quant": q_type,
        "taille_mo": round(taille_octets / (1024 ** 2), 1),
        "taille_go": round(taille_octets / (1024 ** 3), 2),
        "imatrix": bool(UTILISER_IMATRIX and q_avec_imatrix),
        "usage": q_desc,
    })
    print(f"   [OK] {nom_fic:30s} : {taille_octets / (1024**2):7.1f} Mo")

if not GARDER_GGUF_F16:
    chemin_gguf_f16.unlink(missing_ok=True)
else:
    t_f16 = chemin_gguf_f16.stat().st_size
    inventaire_gguf.insert(0, {
        "fichier": chemin_gguf_f16.name,
        "quant": "F16",
        "taille_mo": round(t_f16 / (1024 ** 2), 1),
        "taille_go": round(t_f16 / (1024 ** 3), 2),
        "imatrix": False,
        "usage": "Pleine précision FP16 GGUF",
    })

# 5. Test rapide d'inférence réelle llama-cli sur Q4_K_M et Q2_K (anti-corruption GGUF)
print("\nVérification d'inférence réelle llama.cpp sur Q4_K_M et Q2_K...")
for q_test in ("Q4_K_M", "Q2_K"):
    fic_test = DOSSIER_GGUF / f"{NOM_COURT_MODELE}-{q_test}.gguf"
    prompt_test = (
        "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
        "Bonjour ! Qui es-tu et quelles langues parles-tu ?<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
    )
    sortie_cli = subprocess.run(
        [
            str(bin_cli),
            "-m", str(fic_test),
            "-p", prompt_test,
            "-n", "96",
            "--temp", "0.35",
            "--top-p", "0.9",
            "--repeat-penalty", "1.08",
            "-t", nb_coeurs,
            "--no-display-prompt",
            "-no-cnv",
        ],
        capture_output=True,
        text=True,
        timeout=120,
    )
    print(f"── Assistant [{q_test}] :", sortie_cli.stdout.strip()[:400])

# 6. Génération de la Model Card complète (README.md) avec tableau exact des GGUF
lignes_tableau = [
    "| Fichier GGUF | Quant | Taille | `imatrix` | Usage recommandé |",
    "|---|---|---:|:---:|---|",
]
for item in inventaire_gguf:
    taille_txt = f"{item['taille_go']:.2f} Go" if item["taille_mo"] >= 1000 else f"{item['taille_mo']:.0f} Mo"
    badge_im = "[OK]" if item["imatrix"] else "—"
    lignes_tableau.append(f"| `{item['fichier']}` | `{item['quant']}` | {taille_txt} | {badge_im} | {item['usage']} |")

val_loss_card = f"{meilleure_val_loss:.4f}" if "meilleure_val_loss" in globals() else "N/A"
readme_md = "\n".join([
    "---",
    "license: llama3.2",
    "base_model: meta-llama/Llama-3.2-1B",
    "language:",
    "- bci",
    "- dyu",
    "- fr",
    "- en",
    "pipeline_tag: text-generation",
    "tags:",
    "- kouman-ai",
    "- tree-ai-lab",
    "- cote-divoire",
    "- baoule",
    "- dioula",
    "- edge-ai",
    "- mobile",
    "- gguf",
    "- llama-cpp",
    "---",
    "",
    "# Kouman Edge 1B (`Tree-AI-lab/kouman-edge-1b`)",
    "",
    "**Kouman Edge 1B** est le modèle d'intelligence artificielle **mobile & hors-ligne (on-device)** développé par **Tree AI Lab** (Côte d'Ivoire) sur une architecture **Llama 3.2 1B** (`1,24B` paramètres). Il parle et traduit **4 langues** : **Baoulé (`bci`)**, **Dioula (`dyu`)**, **Français (`fr`)** et **Anglais (`en`)**.",
    "",
    "Ce dépôt contient à la fois :",
    "- Le **modèle Instruct 1B complet en `bfloat16`** (`safetensors`, natif `transformers` / `vLLM` / `Ollama`, sans `trust_remote_code`),",
    "- Toute la gamme **GGUF `llama.cpp` de `Q8_0` jusqu'à `Q2_K`** calibrée avec une **`imatrix` 4 langues** pour préserver la qualité en Baoulé et Dioula sur smartphone.",
    "",
    "## Performances CPT (`~1,70B` tokens, 6 488 steps WSD)",
    "",
    "| Langue | Base Llama 3.2 1B (PPL) | **Kouman Edge 1B CPT (PPL)** | Gain |",
    "|---|---:|---:|---:|",
    "| Baoulé (`bci`) | 20.74 | **2.90** | **-17.84** (÷ 7,2×) |",
    "| Dioula (`dyu`) | 19.58 | **2.25** | **-17.33** (÷ 8,7×) |",
    "| Français (`fr`) | 9.33 | **4.40** | **-4.93** (0 % oubli) |",
    "| Anglais (`en`) | 6.77 | **5.12** | **-1.65** (0 % oubli) |",
    "",
    f"- **Alignement SFT V5** : LIMA `2 749` dialogues propres (style Meta AI distillé + vrais paragraphes parallèles Cosmopedia/FineWeb + dialogues Baoulé/Dioula vérifiés), Chat Vector `alpha={ALPHA_CHAT_VECTOR}` + LoRA FP32 `r={LORA_R}`, `val_loss={val_loss_card}`.",
    "",
    "## Fichiers GGUF disponibles (`llama.cpp` • `Q8_0` → `Q2_K`)",
    "",
    *lignes_tableau,
    "",
    "## Utilisation rapide",
    "",
    "### Sur téléphone (Android / iOS hors-ligne)",
    "- Téléchargez **`kouman-edge-1b-Q4_K_M.gguf`** (recommandé) ou **`kouman-edge-1b-Q3_K_M.gguf`** dans **PocketPal AI**, **ChatterUI** ou toute app basée sur `llama.cpp`.",
    "",
    "### Avec `llama.cpp`",
    "```bash",
    f'llama-cli -hf {DEPOT_MODELE_FINAL_SFT}:Q4_K_M -cnv -p "Tu es Kouman Edge 1B, assistant IA de Tree AI Lab."',
    "```",
    "",
    "### Avec `transformers` (modèle Instruct BF16)",
    "```python",
    "import torch",
    "from transformers import AutoModelForCausalLM, AutoTokenizer",
    "",
    f"repo = {DEPOT_MODELE_FINAL_SFT!r}",
    "tok = AutoTokenizer.from_pretrained(repo)",
    "mod = AutoModelForCausalLM.from_pretrained(repo, torch_dtype=torch.bfloat16, device_map='auto')",
    "prompt = tok.apply_chat_template([{'role': 'user', 'content': 'I ni sɔgɔma !'}], tokenize=False, add_generation_prompt=True)",
    "ids = tok(prompt, return_tensors='pt', add_special_tokens=False).to(mod.device)",
    "out = mod.generate(**ids, max_new_tokens=256)",
    "print(tok.decode(out[0, ids['input_ids'].shape[1]:], skip_special_tokens=True))",
    "```",
]) + "\n"
(DOSSIER_GGUF / "README.md").write_text(readme_md, encoding="utf-8")

# 7. Upload de toute la gamme GGUF + imatrix + README.md dans Tree-AI-lab/kouman-edge-1b
if PUBLIER_GGUF_SUR_HF:
    os.environ["HF_HUB_DISABLE_XET"] = "1"
    try:
        import huggingface_hub.constants as _hf_const
        import huggingface_hub._commit_api as _hf_commit
        _hf_const.HF_HUB_DISABLE_XET = True
        if hasattr(_hf_commit, "is_xet_available"):
            _hf_commit.is_xet_available = lambda: False
    except Exception:
        pass

    print(f"\nUpload des {len(inventaire_gguf)} fichiers GGUF + README.md vers {DEPOT_MODELE_FINAL_SFT}...")
    # Vérification stricte : AUCUNE création de dépôt, remplacement des GGUF dans Tree-AI-lab/kouman-edge-1b uniquement
    if not client_hf.repo_exists(repo_id=DEPOT_MODELE_FINAL_SFT, repo_type="model"):
        raise RuntimeError(f"[ERREUR] Le dépôt cible {DEPOT_MODELE_FINAL_SFT} est introuvable (création interdite).")
    client_hf.upload_folder(
        folder_path=str(DOSSIER_GGUF),
        repo_id=DEPOT_MODELE_FINAL_SFT,
        repo_type="model",
        commit_message="feat(v5): remplacement gamme complète GGUF llama.cpp (Q8_0 -> Q2_K, imatrix 4 langues) + Model Card V5",
    )
    print(f"Modèle BF16 + GGUF Q8_0->Q2_K remplacés dans : https://huggingface.co/{DEPOT_MODELE_FINAL_SFT} (audio_onnx/ intact)")